# Sprint 2: Smog Policy Assistant

In [1]:
import sys
import os

## 1. Tools & Retrieval (rag.py, forecast_tool.py)

In [2]:
from __future__ import annotations
import math
import re
from collections import Counter
from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path
ROOT = Path(".")
DOCS_DIR = ROOT / "docs"
INJECTION_PATTERNS = [
    r"ignore (all )?(previous|prior|above) (instructions|rules)", r"system (note|prompt|message)",
    r"do not (mention|cite|tell)", r"from now on", r"you are now", r"disregard",
    r"always respond", r"regardless of the actual",
]
UNOFFICIAL_WEIGHT = 0.5
ROMAN_URDU = {
    "kal": "tomorrow", "aaj": "today", "parson": "day after tomorrow", "agle": "next",
    "school": "school", "schools": "school", "skool": "school", "sakool": "school",
    "band": "close closure", "bund": "close closure", "chutti": "close closure", "chhutti": "close closure",
    "khulay": "open", "khulenge": "open", "aadha": "half", "adha": "half",
    "bachay": "children", "bachon": "children", "bachey": "children", "bache": "children", "bacchon": "children",
    "bachy": "children", "buzurg": "elderly", "boorhay": "elderly", "budhe": "elderly",
    "hamla": "pregnant", "hamila": "pregnant", "hamilah": "pregnant", "hamal": "pregnant",
    "dama": "asthma", "saans": "breath breathing", "sans": "breath breathing", "khansi": "cough",
    "seena": "chest", "dard": "pain",
    "sehat": "health", "sehet": "health", "hawa": "air", "aloodgi": "pollution", "aaloodgi": "pollution",
    "dhund": "smog", "dhuwan": "smog smoke", "dhuan": "smog smoke", "smog": "smog",
    "mask": "mask", "maask": "mask", "naqab": "mask",
    "gaari": "vehicle car", "gari": "vehicle car", "gaadi": "vehicle car", "gadi": "vehicle car",
    "garion": "vehicle car", "gaariyon": "vehicle car", "number": "number plate",
    "aspatal": "hospital", "haspatal": "hospital", "hospital": "hospital",
    "bahar": "outdoor outside", "baahar": "outdoor outside", "ghar": "indoor home", "andar": "indoor",
    "khel": "sports play", "khelna": "sports play", "varzish": "exercise", "warzish": "exercise",
    "daurna": "exercise running", "sair": "walk outdoor",
    "khatarnak": "hazardous", "khatarnaak": "hazardous", "mehfooz": "safe", "mahfooz": "safe",
    "qanoon": "policy rule", "qawaneen": "policy rule", "pabandi": "restriction ban",
    "ijazat": "allowed", "bhatta": "brick kiln", "bhattay": "brick kiln", "jalana": "burning",
    "fasal": "crop", "kachra": "waste", "taamir": "construction", "tameer": "construction",
    "dawai": "medication", "dawa": "medication", "safar": "travel transport",
    "metro": "metrobus metro", "kiraya": "fare", "tadabeer": "precautions",
    "ehtiyat": "precautions", "ehtiyaat": "precautions", "hidayat": "guidance advice",
    "elaqa": "area", "ilaqa": "area", "ilaqe": "area", "shehar": "city",
}
STOPWORDS = set("""a an the is are was were be been being am do does did of in on at to for from by with
and or but if then than that this these those it its as into about over under up down out what which who
whom whose when where why how can could should would will shall may might must i me my we our you your he
she they them their there here please tell kya hai hain ka ki ke ko se mein me main aur ya bhi hi to tha
thi the hoga hogi honge karna karein chahiye kar raha rahi kaise kaisa kitna kitni kab kahan jab nahi na
""".split())
@dataclass
class Doc:
    doc_id: str
    title: str
    authority: str
    published_date: str
    status: str
    supersedes: str | None
    superseded_by: str | None
    body: str
    security_flags: list[str] = field(default_factory=list)
    @property
    def is_current(self) -> bool:
        return self.status.lower() == "current"
    @property
    def is_official(self) -> bool:
        return "unofficial" not in self.authority.lower()
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    section: str
    text: str
    tokens: list[str]
def tokenize(text: str) -> list[str]:
    raw = re.findall(r"[a-z0-9]+(?:\.[0-9]+)?", text.lower().replace("pm2.5", "pm25"))
    out = []
    for t in raw:
        out.extend(ROMAN_URDU.get(t, t).split())
    norm = []
    for t in out:
        if t in STOPWORDS:
            continue
        if len(t) > 4 and t.endswith("ies"):
            t = t[:-3] + "y"
        elif len(t) > 3 and t.endswith("s") and not t.endswith("ss"):
            t = t[:-1]
        norm.append(t)
    return norm
def _parse_front_matter(raw: str) -> tuple[dict, str]:
    m = re.search(r"^---\s*\n(.*?)\n---\s*\n", raw, re.S | re.M)
    if not m:
        return {}, raw
    meta = {}
    for line in m.group(1).splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            meta[k.strip()] = v.strip()
    body = (raw[:m.start()] + "\n" + raw[m.end():]).strip()
    return meta, body
def sanitize(body: str) -> tuple[str, list[str]]:
    flags = []
    if re.search(r"<!--.*?-->", body, re.S):
        flags.append("hidden HTML comment removed")
        body = re.sub(r"<!--.*?-->", "", body, flags=re.S)
    kept = []
    for line in body.splitlines():
        if any(re.search(p, line, re.I) for p in INJECTION_PATTERNS):
            flags.append(f"instruction-like line removed: {line.strip()[:80]}")
            continue
        kept.append(line)
    return "\n".join(kept), flags
def _chunks_for(doc: Doc) -> list[Chunk]:
    chunks, section, para = [], doc.title, []
    def emit(text):
        text = re.sub(r"\*\*|__|`", "", text).strip()
        if len(text) < 15:
            return
        cid = f"{doc.doc_id}#{len(chunks) + 1}"
        chunks.append(Chunk(cid, doc.doc_id, section, text,
                            tokenize(f"{doc.title} {section} {text}")))
    for line in doc.body.splitlines():
        s = line.strip()
        if not s:
            if para:
                emit(" ".join(para)); para = []
            continue
        if s.startswith("#"):
            if para:
                emit(" ".join(para)); para = []
            section = s.lstrip("#").strip()
            continue
        if re.fullmatch(r"\*\*[^*]+\*\*", s) and len(s) < 60:   
            if para:
                emit(" ".join(para)); para = []
            section = s.strip("*").strip()
            continue
        if s.endswith(":") and not s.startswith(("- ", "* ")):     
            if para:
                emit(" ".join(para)); para = []
            section = re.sub(r"\*\*", "", s[:-1]).strip()
            continue
        if s.startswith(("- ", "* ")):
            if para:
                emit(" ".join(para)); para = []
            emit(s[2:])
            continue
        para.append(s.lstrip("> "))
    if para:
        emit(" ".join(para))
    return chunks
@lru_cache(maxsize=1)
def load_corpus() -> tuple[dict[str, Doc], list[Chunk]]:
    docs, chunks = {}, []
    for path in sorted(DOCS_DIR.glob("DOC-*.md")):
        raw = path.read_text(encoding="utf-8", errors="replace")
        meta, body = _parse_front_matter(raw)
        body, flags = sanitize(body)
        doc = Doc(
            doc_id=meta.get("document_id", path.stem[:6]),
            title=meta.get("title", path.stem),
            authority=meta.get("authority", "unknown"),
            published_date=meta.get("published_date", ""),
            status=meta.get("status", "unknown"),
            supersedes=meta.get("supersedes"),
            superseded_by=meta.get("superseded_by"),
            body=body,
            security_flags=flags,
        )
        docs[doc.doc_id] = doc
        chunks.extend(_chunks_for(doc))
    for d in docs.values():
        if d.supersedes and d.supersedes in docs:
            old = docs[d.supersedes]
            old.status, old.superseded_by = "superseded", d.doc_id
    return docs, chunks
class BM25:
    def __init__(self, chunks: list[Chunk], k1=1.4, b=0.75):
        self.chunks, self.k1, self.b = chunks, k1, b
        self.tf = [Counter(c.tokens) for c in chunks]
        self.len = [len(c.tokens) for c in chunks]
        self.avg = sum(self.len) / len(self.len)
        df = Counter(t for c in chunks for t in set(c.tokens))
        n = len(chunks)
        self.idf = {t: math.log(1 + (n - f + 0.5) / (f + 0.5)) for t, f in df.items()}
    def score(self, q_tokens: list[str], i: int) -> float:
        s, tf, dl = 0.0, self.tf[i], self.len[i]
        for t in set(q_tokens):
            if t in tf:
                f = tf[t]
                s += self.idf[t] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * dl / self.avg))
        return s
@lru_cache(maxsize=1)
def _index() -> BM25:
    return BM25(load_corpus()[1])
@dataclass
class Hit:
    chunk: Chunk
    doc: Doc
    score: float
def retrieve(query: str, k: int = 8, include_superseded: bool = False,
             doc_boost: dict[str, float] | None = None, only_docs: set[str] | None = None) -> list[Hit]:
    docs, chunks = load_corpus()
    idx = _index()
    q = tokenize(query)
    doc_boost = doc_boost or {}
    hits = []
    for i, c in enumerate(chunks):
        d = docs[c.doc_id]
        if only_docs is not None and c.doc_id not in only_docs:
            continue
        if not d.is_current and not include_superseded:
            continue
        s = idx.score(q, i)
        s *= (1.0 if d.is_official else UNOFFICIAL_WEIGHT)
        s += doc_boost.get(c.doc_id, 0.0) * (1 if s > 0 else 0.5)
        if s > 0:
            hits.append(Hit(c, d, s))
    hits.sort(key=lambda h: (-h.score, h.doc.published_date))
    return hits[:k]
if __name__ == "__main__":
    docs, chunks = load_corpus()
    print(f"{len(docs)} docs, {len(chunks)} chunks")
    for d in docs.values():
        print(f"  {d.doc_id} [{d.status}{' -> ' + d.superseded_by if d.superseded_by else ''}] "
              f"{d.published_date} {d.authority[:40]} flags={d.security_flags}")
    for q in ["When do schools close because of smog?", "kya bachon ko school bhejna chahiye?",
              "Which mask should I wear?", "odd even gaari pabandi"]:
        print("\nQ:", q)
        for h in retrieve(q, k=4):
            print(f"  {h.score:5.2f} {h.chunk.chunk_id:8} {h.chunk.text[:90]}")


from __future__ import annotations
import re
from datetime import date
from functools import lru_cache
from pathlib import Path
import pandas as pd
ROOT = Path(".")
TEAM_TABLE = ROOT / "data/forecast_table.csv"            
REFERENCE_TABLE = ROOT / "data/reference_forecast.csv"   
HAZARD = 165.0
AREA_ALIASES = {
    "gulberg": "Gulberg",
    "dha": "DHA", "defence": "DHA", "defense": "DHA", "d.h.a": "DHA",
    "johar town": "Johar Town", "johar": "Johar Town", "joher town": "Johar Town",
    "model town": "Model Town",
    "wagah": "Wagah", "wagha": "Wagah", "wagah border": "Wagah",
    "thokar niaz baig": "Thokar Niaz Baig", "thokar": "Thokar Niaz Baig", "niaz baig": "Thokar Niaz Baig",
    "allama iqbal town": "Allama Iqbal Town", "iqbal town": "Allama Iqbal Town", "aiqbal town": "Allama Iqbal Town",
    "shadman": "Shadman",
    "cantt": "Cantt", "cantonment": "Cantt", "lahore cantt": "Cantt",
    "faisal town": "Faisal Town",
    "township": "Township",
    "walled city": "Walled City", "androon shehar": "Walled City", "androon lahore": "Walled City",
    "andaroon shehar": "Walled City", "old city": "Walled City",
    "raiwind road": "Raiwind Road", "raiwind": "Raiwind Road",
    "bahria town": "Bahria Town", "bahria": "Bahria Town",
    "airport road": "Airport Road", "airport": "Airport Road",
}
@lru_cache(maxsize=1)
def _meta() -> pd.DataFrame:
    return pd.read_csv(ROOT / "weather/sensor_metadata.csv")
def covered_areas() -> list[str]:
    return _meta().area.tolist()
def area_to_sensor(area: str) -> str:
    m = _meta()
    return m.loc[m.area == area, "sensor_id"].iloc[0]
def resolve_location(location) -> str | None:
    if not isinstance(location, str):
        return None
    loc = re.sub(r"\s+", " ", location.strip().lower())
    loc = re.sub(r",?\s*lahore$", "", loc).strip()
    m = _meta()
    if re.fullmatch(r"s(0[1-9]|1[0-5])", loc):
        return m.loc[m.sensor_id == loc.upper(), "area"].iloc[0]
    for area in m.area:
        if loc == area.lower():
            return area
    return AREA_ALIASES.get(loc)
@lru_cache(maxsize=1)
def _table() -> tuple[pd.DataFrame | None, str]:
    if TEAM_TABLE.exists():
        return pd.read_csv(TEAM_TABLE), "team_model"
    if REFERENCE_TABLE.exists():
        return pd.read_csv(REFERENCE_TABLE), "reference_fallback"
    return None, "team_model"
def supported_window() -> tuple[str, str] | None:
    t, _ = _table()
    if t is None:
        return None
    return t.target_date.min(), t.target_date.max()
def forecast_with_reason(location, target_date) -> tuple[dict, str]:
    table, source = _table()
    area = resolve_location(location)
    td = target_date.isoformat() if isinstance(target_date, date) else str(target_date).strip()
    result = {
        "location": area or (str(location) if location is not None else ""),
        "target_date": td,
        "pm25": None,
        "hazardous": None,
        "status": "unavailable",
        "source": source,
    }
    if table is None:
        return result, "no forecast table available"
    if area is None:
        return result, f"location '{location}' is not one of the 15 covered Lahore areas"
    try:
        parsed = date.fromisoformat(td)
    except ValueError:
        return result, f"target_date '{td}' is not a valid YYYY-MM-DD date"
    result["target_date"] = parsed.isoformat()
    row = table[(table.area == area) & (table.target_date == parsed.isoformat())]
    if len(row) != 1:
        lo, hi = supported_window()
        return result, f"{parsed.isoformat()} is outside the supported forecast window {lo} to {hi}"
    pm = float(row.predicted_pm25.iloc[0])
    result.update(pm25=round(pm, 1), hazardous=bool(pm >= HAZARD), status="ok")
    return result, "ok"
def forecast(location, target_date) -> dict:
    return forecast_with_reason(location, target_date)[0]
if __name__ == "__main__":
    for args in [("Gulberg", "2026-10-29"), ("s12", "2026-11-07"), ("Karachi", "2026-10-30"),
                 ("DHA", "2026-12-01"), ("Model Town", "tomorrow")]:
        print(args, "->", forecast(*args))


13 docs, 77 chunks
  DOC-01 [current] 2026-10-15 Punjab Environmental Protection & Climat flags=[]
  DOC-02 [superseded -> DOC-03] 2025-11-01 Punjab School Education Department flags=[]
  DOC-03 [current] 2026-09-20 Punjab School Education Department flags=[]
  DOC-04 [current] 2026-10-01 Punjab Environmental Protection & Climat flags=[]
  DOC-05 [current] 2026-10-10 Punjab Transport Department flags=[]
  DOC-06 [current] 2026-10-20 Lahore Smog Intelligence Challenge — Org flags=[]
  DOC-07 [current] 2026-10-18 Local Resident Welfare Association (unof flags=['hidden HTML comment removed']
  DOC-08 [current] 2026-10-05 Punjab Environmental Protection & Climat flags=[]
  DOC-09 [current] 2026-10-12 Punjab Health Department flags=[]
  DOC-10 [current] 2026-10-08 Punjab Transport Department flags=[]
  DOC-11 [current] 2026-10-14 Punjab Health Department flags=[]
  DOC-12 [current] 2026-10-16 Punjab Environmental Protection & Climat flags=[]
  DOC-13 [current] 2026-10-20 Lahore Smog Intelli

('Gulberg', '2026-10-29') -> {'location': 'Gulberg', 'target_date': '2026-10-29', 'pm25': 135.1, 'hazardous': False, 'status': 'ok', 'source': 'team_model'}
('s12', '2026-11-07') -> {'location': 'Walled City', 'target_date': '2026-11-07', 'pm25': 157.3, 'hazardous': False, 'status': 'ok', 'source': 'team_model'}
('Karachi', '2026-10-30') -> {'location': 'Karachi', 'target_date': '2026-10-30', 'pm25': None, 'hazardous': None, 'status': 'unavailable', 'source': 'team_model'}
('DHA', '2026-12-01') -> {'location': 'DHA', 'target_date': '2026-12-01', 'pm25': None, 'hazardous': None, 'status': 'unavailable', 'source': 'team_model'}
('Model Town', 'tomorrow') -> {'location': 'Model Town', 'target_date': 'tomorrow', 'pm25': None, 'hazardous': None, 'status': 'unavailable', 'source': 'team_model'}


## 2. Assistant Core (assistant.py)

In [3]:
from __future__ import annotations
import hashlib
import json
import os
import re
import sys
from dataclasses import dataclass, field
from datetime import date, timedelta
REFERENCE_TODAY = date.fromisoformat(os.getenv("SMOG_TODAY", "2026-10-28"))  
VALIDATION_MAE = 13.4  
HAZARD = 165.0
BAND_SECTION = {"hazardous": r"hazardous range", "unhealthy": r"between 100", "low": r"below 100"}
OTHER_CITIES = [
    "karachi", "islamabad", "quetta", "peshawar", "multan", "faisalabad", "rawalpindi", "gujranwala",
    "sialkot", "hyderabad", "sargodha", "bahawalpur", "sukkur", "kasur", "sheikhupura", "abbottabad",
    "murree", "gujrat", "sahiwal", "okara", "jhelum", "mardan", "larkana", "gwadar", "delhi",
    "new delhi", "amritsar", "mumbai", "dubai", "london", "kabul", "chitral", "swat", "skardu",
]
NON_COVERED_LAHORE = [
    "shahdara", "gulshan-e-ravi", "gulshan e ravi", "wapda town", "valencia", "garden town", "samanabad",
    "ichhra", "mughalpura", "baghbanpura", "sabzazar", "askari", "lake city", "eme", "green town",
    "kot lakhpat", "shalimar", "data darbar", "anarkali", "mall road", "gulberg iii",  
    "paragon city", "dha phase", "state life", "nishtar town", "harbanspura", "chung", "manawan",
]
NON_COVERED_LAHORE.remove("gulberg iii")
NON_COVERED_LAHORE.remove("dha phase")
MONTHS = {m: i + 1 for i, m in enumerate(
    ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"])}
MONTH_RE = r"(jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|june?|july?|aug(?:ust)?|sept?(?:ember)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)"
WEEKDAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
TOPICS = {
    "school": (r"school|skool|class(es)?\b|assembl|recess|student|parhai|taleem|college", ["DOC-03"]),
    "vehicle": (r"odd|even|vehicle|car\b|cars\b|gaa?r[iy]|gaadi|gadi|number plate|drive|driving", ["DOC-05"]),
    "construction": (r"construct|brick|kiln|bhatt|burn|industr|factory|demolition|excavat|taamir|tameer|crop residue|kachra|waste", ["DOC-08"]),
    "hospital": (r"hospital|opd|nebuli|emergency|aspatal|haspatal|doctor|clinic", ["DOC-09"]),
    "transport": (r"metro|orange line|public transport|bus\b|park.and.ride|fare|kiraya|commute|safar", ["DOC-10"]),
    "mask": (r"mask|maask|n95|kn95|purifier|hepa|filter|naqab", ["DOC-12", "DOC-01"]),
    "vulnerable": (r"child|kid|bach|pregnan|hamil|hamla|elder|old people|buzurg|asthma|dama|copd|heart|senior|infant|baby|65", ["DOC-11", "DOC-01"]),
    "health": (r"health|safe|outdoor|outside|exercise|jog|run\b|running|walk|sair|bahar|baahar|varzish|warzish|precaution|ehtiya|symptom|cough|khansi|saans|breath|window|khel|sport|play", ["DOC-01"]),
    "sensor": (r"sensor|network|aqi|-999|timezone|utc|flat.?line|data quality|station", ["DOC-06", "DOC-04"]),
    "coverage": (r"coverage|covered|which areas|kaun se ilaq|cities|scope", ["DOC-13"]),
    "community": (r"tip|community|bulletin|resident|neighbo|plant|damp cloth|carpool", ["DOC-07"]),
    "faq": (r"what is pm|pm2\.?5 kya|why .*(smog|worse)|october|december|who issues|alert", ["DOC-04"]),
}
ADVICE_RE = (r"should|can i|can my|can we|safe|advice|advise|precaution|recommend|what to do|kya kar|"
             r"chahiye|bhej|send|go out|jog|run|exercise|walk|play|mask|school|open|close|band|khul|allowed|"
             r"ijazat|odd|even|hospital|guidance|hidayat|tips|protect|bach(ao|ne)|outdoor|outside|bahar|"
             r"restriction|pabandi|rule|policy|kiln|construction|metro|transport|children|kids|elderly|"
             r"pregnan|asthma|tadabeer|ehtiya")
FORECAST_RE = (r"forecast|predict|pm\s?2\.?5|pm25|aqi|air quality|smog level|pollution level|level|"
               r"how bad|kitna|kitni|kaisa|kaisi|hoga|hogi|expected|tomorrow|kal\b|parson|hazardous|"
               r"khatarna+k|reading")
OLD_RE = r"\b(old|older|previous|earlier|former|2023|2025|version 1|v1|purana|purani|pehle|superseded|outdated|last year)\b"
WINDOW_RE = r"week|hafta|hafte|next (few|10|ten) days|coming days|agle (kuch )?din|all (the )?days|whole window"
ROMAN_MARKERS = {"kya", "hai", "hain", "ka", "ki", "ke", "mein", "kal", "aaj", "karna", "chahiye", "nahi",
                 "ko", "se", "aur", "hoga", "hogi", "kitna", "kitni", "bachon", "kaise", "kahan", "kab",
                 "bhejna", "bhejein", "jayein", "jana", "sakte", "sakta", "sakti", "hum", "mujhe", "apne",
                 "wala", "wali", "liye", "raha", "rahi", "kar", "parson", "batao", "bataein", "khatarnak"}
AREA_NAMES = sorted(set(AREA_ALIASES) | {a.lower() for a in covered_areas()}, key=len, reverse=True)
@dataclass
class Ctx:
    question: str
    q: str
    roman: bool
    retrieved: set = field(default_factory=set)
    forecast_called: bool = False
    forecasts: list = field(default_factory=list)
    numbers: set = field(default_factory=set)       
    def call_forecast(self, area: str, d: date | str):
        self.forecast_called = True
        res, reason = forecast_with_reason(area, d.isoformat() if isinstance(d, date) else d)
        self.forecasts.append(res)
        if res["pm25"] is not None:
            self.numbers.add(f"{res['pm25']:.1f}")
        return res, reason
    def chunks(self, doc_id: str, query: str = "", section_has: str | None = None,
               k: int = 3, include_superseded: bool = False) -> list[Hit]:
        hits = retrieve(query or doc_id, k=50, include_superseded=include_superseded,
                            only_docs={doc_id}, doc_boost={doc_id: 0.01})
        if not hits:  
            docs, chunks = load_corpus()
            d = docs[doc_id]
            if not d.is_current and not include_superseded:
                return []
            hits = [Hit(c, d, 0.0) for c in chunks if c.doc_id == doc_id]
        if section_has:
            hits = [h for h in hits if re.search(section_has, h.chunk.section, re.I)]
        hits = hits[:k]
        self.retrieved |= {h.doc.doc_id for h in hits}
        return hits
def _split_id(question: str) -> tuple[str, str]:
    m = re.match(r"^\s*\**\s*(Q[\s\-_]?\d{1,3})\s*\**\s*[:.)\-]\s*(.*)$", question, re.S | re.I)
    if m:
        return re.sub(r"[\s\-_]", "", m.group(1)).upper(), m.group(2).strip()
    return "Q-" + hashlib.sha1(question.encode("utf-8")).hexdigest()[:8], question.strip()
def _is_roman_urdu(q: str) -> bool:
    toks = re.findall(r"[a-z]+", q)
    return sum(t in ROMAN_MARKERS for t in toks) >= 2
def _find_areas(q: str) -> list[str]:
    found, taken = [], [False] * len(q)
    for name in AREA_NAMES:
        for m in re.finditer(rf"(?<![a-z]){re.escape(name)}(?![a-z])", q):
            if any(taken[m.start():m.end()]):
                continue
            for i in range(m.start(), m.end()):
                taken[i] = True
            area = resolve_location(name)
            if area:
                found.append((m.start(), area))
    for m in re.finditer(r"\bs(0[1-9]|1[0-5])\b", q):
        area = resolve_location(m.group(0))
        found.append((m.start(), area))
    out = []
    for _, a in sorted(found):
        if a not in out:
            out.append(a)
    return out
def _find_other_places(q: str) -> list[str]:
    hits = []
    for c in OTHER_CITIES + NON_COVERED_LAHORE:
        if re.search(rf"(?<![a-z]){re.escape(c)}(?![a-z])", q):
            hits.append(c.title())
    return hits
def _find_dates(q: str) -> tuple[list[date], list[str]]:
    found, notes = [], []
    for m in re.finditer(r"\b(20\d{2})-(\d{1,2})-(\d{1,2})\b", q):
        found.append((m.start(), _safe_date(int(m.group(1)), int(m.group(2)), int(m.group(3)))))
    for m in re.finditer(r"\b(\d{1,2})/(\d{1,2})/(20\d{2})\b", q):  
        found.append((m.start(), _safe_date(int(m.group(3)), int(m.group(2)), int(m.group(1)))))
    for m in re.finditer(rf"\b(\d{{1,2}})(?:st|nd|rd|th)?\s+(?:of\s+)?{MONTH_RE}\.?(?:,?\s+(20\d{{2}}))?", q):
        found.append((m.start(), _safe_date(int(m.group(3) or REFERENCE_TODAY.year),
                                            MONTHS[m.group(2)[:3]], int(m.group(1)))))
    for m in re.finditer(rf"\b{MONTH_RE}\.?\s+(\d{{1,2}})(?:st|nd|rd|th)?\b(?:,?\s+(20\d{{2}}))?", q):
        found.append((m.start(), _safe_date(int(m.group(3) or REFERENCE_TODAY.year),
                                            MONTHS[m.group(1)[:3]], int(m.group(2)))))
    rel = [(r"day after tomorrow|parson|parso\b", 2), (r"\btomorrow\b|\bkal\b|agle din|next day", 1),
           (r"\btoday\b|\baaj\b|tonight", 0)]
    for pat, off in rel:
        m = re.search(pat, q)
        if m:
            d = REFERENCE_TODAY + timedelta(days=off)
            found.append((m.start(), d))
            notes.append(f"'{m.group(0)}' is read relative to the latest available data day "
                         f"({REFERENCE_TODAY.isoformat()}), i.e. {d.isoformat()}")
            break
    for i, wd in enumerate(WEEKDAYS):
        m = re.search(rf"\b{wd}\b", q)
        if m:
            delta = (i - REFERENCE_TODAY.weekday()) % 7 or 7
            d = REFERENCE_TODAY + timedelta(days=delta)
            found.append((m.start(), d))
            notes.append(f"'{wd.title()}' is read as the next {wd.title()} after {REFERENCE_TODAY.isoformat()}, "
                         f"i.e. {d.isoformat()}")
    out = []
    for _, d in sorted(found, key=lambda x: x[0]):
        if d not in out:
            out.append(d)
    return out, notes
def _safe_date(y, m, d):
    try:
        return date(y, m, d)
    except ValueError:
        return f"{y:04d}-{m:02d}-{d:02d}"  
def _topics(q: str) -> list[str]:
    return [t for t, (pat, _) in TOPICS.items() if re.search(pat, q)]
def _bullet(h: Hit, label_section=True) -> str:
    txt = h.chunk.text.strip()
    sec = h.chunk.section
    if label_section and sec and sec != h.doc.title and not sec.lower().startswith(h.doc.title.lower()[:20])            and len(sec) < 40:
        txt = f"{sec}: {txt}"
    return f"- {txt} [{h.doc.doc_id}]"
def _doc_header(d: Doc) -> str:
    status = "current" if d.is_current else f"SUPERSEDED by {d.superseded_by}"
    note = " - unofficial community tips, not official guidance" if not d.is_official else ""
    return f"{d.doc_id} \"{d.title}\" ({d.authority}, {d.published_date}, {status}{note})"
def _band(pm: float) -> str:
    return "hazardous" if pm >= HAZARD else ("unhealthy" if pm >= 100 else "low")
def _window_text() -> str:
    w = supported_window()
    return f"{w[0]} to {w[1]}" if w else "no dates (forecast table missing)"
def _forecast_sentence(ctx: Ctx, res: dict, reason: str) -> str:
    area, d = res["location"], res["target_date"]
    if res["status"] != "ok":
        ctx.chunks("DOC-13", "dates outside the supported forecast window out of scope", k=2)
        if ctx.roman:
            return (f"{area} ke liye {d} ka koi forecast dastyab nahi ({reason}). Hum andaza nahi lagate. "
                    f"Yeh system sirf Lahore ke 15 covered ilaqon aur {_window_text()} ki tareekhon ke liye hai [DOC-13].")
        return (f"No forecast is available for {area} on {d} ({reason}). I will not estimate a number. "
                f"Coverage is limited to the 15 named Lahore areas and target dates {_window_text()} [DOC-13].")
    sid = area_to_sensor(res["location"])
    pm, hz = res["pm25"], res["hazardous"]
    prov = "team forecast model" if res["source"] == "team_model" else "organisers' reference forecast"
    if ctx.roman:
        status = ("yeh KHATARNAK (hazardous) satah hai, yani 165 µg/m³ ya us se zyada" if hz else
                  "yeh hazardous had (165 µg/m³) se kam hai")
        return f"Forecast: {area} (sensor {sid}) mein {d} ko andazan PM2.5 {pm:.1f} µg/m³ hai - {status}. (Source: {prov})"
    status = "HAZARDOUS (at or above the 165 µg/m³ threshold)" if hz else "not hazardous (below the 165 µg/m³ threshold)"
    return f"Forecast: {area} (sensor {sid}) on {d}: predicted daily PM2.5 {pm:.1f} µg/m³ - {status}. (Source: {prov})"
def _guidance_for_forecast(ctx: Ctx, res: dict, topics: list[str], area: str) -> list[str]:
    pm, d = res["pm25"], res["target_date"]
    band = _band(pm)
    out = []
    t = set(topics) or {"health"}
    if "school" in t:
        hits = ctx.chunks("DOC-03", "schools half-day close fully outdoor sports decisions evening before", k=6)
        if pm >= 250:
            dec = "full school closure"
        elif pm >= HAZARD:
            dec = "half-day timing (close by 12:00 PM) and suspension of outdoor sports, assemblies and recess"
        else:
            dec = "no half-day timing or closure, because the forecast is below the 165 µg/m³ trigger"
        out.append(f"School rule for this forecast: {dec} [DOC-03].")
        out += [_bullet(h, False) for h in hits if re.search(r"half-day|close fully|evening before|outdoor sports", h.chunk.text)]
    if "vehicle" in t:
        base = date.fromisoformat(d)
        neigh = {}
        for off in (-1, 1):
            r2, _ = ctx.call_forecast(area, base + timedelta(days=off))
            neigh[off] = r2["hazardous"] if r2["status"] == "ok" else None
        consecutive = res["hazardous"] and (neigh[-1] or neigh[1])
        hits = ctx.chunks("DOC-05", "odd even hazardous two consecutive days exemptions lifted", k=6)
        if consecutive:
            out.append("Odd-even rule: this day is forecast hazardous together with an adjacent day, so the "
                       "two-consecutive-day trigger for the odd-even restriction is met [DOC-05].")
        else:
            out.append("Odd-even rule: the restriction needs hazardous forecasts (165 µg/m³ or above) on two or more "
                       "consecutive days; that trigger is not met by this forecast [DOC-05].")
        out += [_bullet(h, False) for h in hits[:3]]
    if "construction" in t:
        hits = ctx.chunks("DOC-08", "open burning construction brick kilns industries hazardous", k=5)
        if res["hazardous"]:
            out.append("Because the forecast is hazardous, these restrictions apply [DOC-08]:")
            out += [_bullet(h, False) for h in hits[:4]]
        else:
            out.append("The construction/industrial restrictions apply only when forecast PM2.5 reaches 165 µg/m³ or above, "
                       "so they are not triggered by this forecast [DOC-08].")
    if "transport" in t:
        hits = ctx.chunks("DOC-10", "metrobus orange line frequency fares park-and-ride", k=4)
        out.append(("Hazardous-day public transport measures [DOC-10]:" if res["hazardous"]
                    else "Public transport smog measures (frequency increase applies on forecast hazardous days) [DOC-10]:"))
        out += [_bullet(h, False) for h in hits[:3]]
    if "hospital" in t:
        hits = ctx.chunks("DOC-09", "extended opd nebulization emergency chest pain", k=4)
        out += [_bullet(h, False) for h in hits[:3]]
    if "mask" in t:
        hits = ctx.chunks("DOC-12", "n95 kn95 masks 15 minutes hepa purifier", k=5)
        out += [_bullet(h, False) for h in hits[:3]]
    if "vulnerable" in t and band == "hazardous":
        q = ctx.q
        groups = []
        if re.search(r"child|kid|bach|infant|baby|student", q): groups.append("Children")
        if re.search(r"elder|old people|buzurg|senior|65", q): groups.append("Elderly")
        if re.search(r"pregnan|hamil|hamla", q): groups.append("Pregnant")
        if re.search(r"asthma|dama|copd", q): groups.append("asthma")
        for g in groups or ["Children", "Elderly"]:
            out += [_bullet(h) for h in ctx.chunks("DOC-11", ctx.q, section_has=g, k=2)]
    if "community" in t:
        hits = ctx.chunks("DOC-07", ctx.q + " tips", k=3)
        if hits:
            out.append("Community tips (unofficial resident association, not official guidance) [DOC-07]:")
            out += [_bullet(h, False) for h in hits]
    if t & {"health", "vulnerable", "mask"} or not (t & {"school", "vehicle", "construction", "transport", "hospital", "community"}):
        sec = BAND_SECTION[band]
        hits = ctx.chunks("DOC-01", ctx.q + " outdoor masks windows sensitive groups", section_has=sec, k=4)
        label = {"hazardous": "Health advice for a hazardous day",
                 "unhealthy": "Health advice for PM2.5 in the 100-164 µg/m³ range",
                 "low": "Health advice for PM2.5 below 100 µg/m³"}[band]
        out.append(f"{label} [DOC-01]:")
        out += [_bullet(h, False) for h in hits]
    return out
def _doc_answer(ctx: Ctx, topics: list[str], asks_old: bool) -> list[str]:
    boost = {}
    for t in topics:
        for i, did in enumerate(TOPICS[t][1]):
            boost[did] = max(boost.get(did, 0), 2.0 - 0.5 * i)
    hits = retrieve(ctx.question, k=12, include_superseded=asks_old, doc_boost=boost)
    if not hits or hits[0].score < 1.5:
        return []
    top = hits[0].score
    hits = [h for h in hits if h.score >= 0.4 * top]
    by_doc: dict[str, list[Hit]] = {}
    for h in hits:
        by_doc.setdefault(h.doc.doc_id, []).append(h)
    for did in list(by_doc):
        d = by_doc[did][0].doc
        if not d.is_current and d.superseded_by:
            repl = ctx.chunks(d.superseded_by, ctx.question, k=4)
            if repl:
                by_doc = {d.superseded_by: repl, **{k: v for k, v in by_doc.items() if k != d.superseded_by}}
    order = sorted(by_doc, key=lambda k: (not by_doc[k][0].doc.is_official, -max(h.score for h in by_doc[k])))
    if any(by_doc[k][0].doc.is_official for k in order):
        order = [k for k in order if by_doc[k][0].doc.is_official or len(order) == 1]
    order = [k for k in order if by_doc[k][0].doc.is_current][:3]
    old_docs = []
    if asks_old:
        docs, _ = load_corpus()
        for k in order:
            old = docs[k].supersedes
            if old and old in docs:
                hits_old = ctx.chunks(old, ctx.question, k=3, include_superseded=True)
                if hits_old:
                    old_docs.append((old, hits_old))
    lines = []
    for did in order:
        d = by_doc[did][0].doc
        ctx.retrieved.add(did)
        lines.append(f"From {_doc_header(d)}:")
        lines += [_bullet(h) for h in by_doc[did][:4]]
    for did, hits_old in old_docs:
        d = hits_old[0].doc
        lines.append(f"Note: {did} is SUPERSEDED by {d.superseded_by} and must not be used for current decisions; "
                     f"{d.superseded_by} above controls. For reference only, the old version said:")
        lines += [_bullet(h) for h in hits_old]
    return lines
def ask(question: str) -> dict:
    qid, text = _split_id(question)
    ctx = Ctx(question=text, q=text.lower(), roman=_is_roman_urdu(text.lower()))
    q = ctx.q
    areas = _find_areas(q)
    others = _find_other_places(q)
    dates, date_notes = _find_dates(q)
    topics = _topics(q)
    asks_old = bool(re.search(OLD_RE, q))
    advice = bool(re.search(ADVICE_RE, q))
    wants_forecast = bool(re.search(FORECAST_RE, q)) or bool(dates)
    mentions_lahore = bool(re.search(r"\blahore\b", q))
    parts: list[str] = []
    for place in others:
        ctx.chunks("DOC-13", "other cities out of scope no forecast available", k=2)
        if ctx.roman:
            parts.append(f"Maazrat, {place} ke liye koi forecast dastyab nahi. Yeh system sirf Lahore ke 15 ilaqon "
                         f"({', '.join(covered_areas())}) aur {_window_text()} ki tareekhon ko cover karta hai, "
                         f"is liye hum andaza nahi lagayenge [DOC-13].")
        else:
            parts.append(f"No forecast is available for {place}: this system covers only 15 Lahore areas "
                         f"({', '.join(covered_areas())}) and target dates {_window_text()}, so I will not "
                         f"estimate a number [DOC-13].")
    if others and not mentions_lahore:
        areas = []  # "DHA Karachi", "Gulberg Islamabad": same names exist in other cities, never guess
    if others and not areas:
        if advice and topics and not wants_forecast:
            parts.append("The supplied policy documents apply to Punjab/Lahore and may not reflect rules elsewhere [DOC-13].")
        return _finish(qid, ctx, parts, date_notes=[])
    fc_dates = [d for d in dates] or ([REFERENCE_TODAY + timedelta(days=1)] if (areas or mentions_lahore) else [])
    if dates and not areas and not others:
        mentions_lahore = True  # system covers Lahore only: "safe tomorrow?" -> Lahore-wide summary
    if not dates and (areas or mentions_lahore) and (wants_forecast or advice):
        if re.search(WINDOW_RE, q):
            lo, hi = supported_window()
            fc_dates = [date.fromisoformat(lo) + timedelta(days=i)
                        for i in range((date.fromisoformat(hi) - date.fromisoformat(lo)).days + 1)]
        else:
            date_notes.append(f"No date was given, so I used the next forecast day, {fc_dates[0].isoformat()} "
                              f"(the day after the latest available data, {REFERENCE_TODAY.isoformat()})")
    doc_only = not areas and not (mentions_lahore and wants_forecast and (dates or not topics))
    if areas and (wants_forecast or advice or dates):
        multi = len(fc_dates) > 3
        for area in areas:
            for d in fc_dates:
                res, reason = ctx.call_forecast(area, d)
                parts.append(_forecast_sentence(ctx, res, reason))
                if res["status"] == "ok" and advice and not multi:
                    parts += _guidance_for_forecast(ctx, res, topics, area)
        if any(f["status"] == "ok" for f in ctx.forecasts) and re.search(r"\baqi\b", q):
            ctx.chunks("DOC-04", "difference between PM2.5 and AQI", k=1)
            parts.append("Note: the forecast is PM2.5 concentration in µg/m³, not AQI; the two are not "
                         "numerically interchangeable [DOC-04].")
        if any(f["status"] == "ok" for f in ctx.forecasts):
            parts.append(f"Limitation: this is a statistical forecast of the daily average; in back-testing its typical "
                         f"error was about ±{VALIDATION_MAE:.0f} µg/m³ and single-day spikes are hard to predict.")
            ctx.numbers.add(f"{VALIDATION_MAE:.0f}")
    elif not areas and mentions_lahore and wants_forecast and not doc_only:
        for d in fc_dates[:3]:
            rows = [ctx.call_forecast(a, d) for a in covered_areas()]
            ok = [r for r, _ in rows if r["status"] == "ok"]
            if not ok:
                parts.append(_forecast_sentence(ctx, rows[0][0] | {"location": "Lahore"}, rows[0][1]))
                continue
            lo_r = min(ok, key=lambda r: r["pm25"]); hi_r = max(ok, key=lambda r: r["pm25"])
            avg = sum(r["pm25"] for r in ok) / len(ok)
            ctx.numbers.add(f"{avg:.1f}")
            hz = [r["location"] for r in ok if r["hazardous"]]
            parts.append(f"Lahore forecast for {d.isoformat() if isinstance(d, date) else d} across the 15 covered areas: "
                         f"lowest {lo_r['pm25']:.1f} µg/m³ ({lo_r['location']}), highest {hi_r['pm25']:.1f} µg/m³ "
                         f"({hi_r['location']}), average {avg:.1f} µg/m³. Hazardous (>=165): "
                         f"{', '.join(hz) if hz else 'none'}. (Source: team forecast model)")
            if advice:
                parts += _guidance_for_forecast(ctx, hi_r, topics, hi_r["location"])
        parts.append("Ask about a specific area (e.g. Gulberg, DHA) for an area-level forecast.")
    if doc_only:
        doc_lines = _doc_answer(ctx, topics, asks_old)
        if doc_lines:
            if ctx.roman:
                parts.append("Official documents ke mutabiq (quotes English mein):")
            parts += doc_lines
            if dates and not areas and (wants_forecast or "school" in topics):
                parts.append("For an area-specific decision, ask about one of the 15 covered Lahore areas with a date "
                             f"between {_window_text()}.")
        elif not parts:
            parts.append("Maazrat, supplied documents mein is sawal ka qabil-e-aitbaar jawab nahi mila." if ctx.roman else
                         "I could not find reliable information on this in the supplied smog documents or forecast, "
                         "so I won't guess. I can help with smog health advice, school/vehicle/industry rules, and "
                         "PM2.5 forecasts for the 15 covered Lahore areas.")
    return _finish(qid, ctx, parts, date_notes)
def _finish(qid: str, ctx: Ctx, parts: list[str], date_notes: list[str]) -> dict:
    if date_notes:
        parts.append("Assumption: " + "; ".join(date_notes) + ".")
    seen, uniq = set(), []
    for p in parts:
        if p and p not in seen:
            seen.add(p)
            uniq.append(p)
    answer = "\n".join(uniq)
    cited = []
    for did in re.findall(r"\[(DOC-\d{2})\]", answer):
        if did not in cited:
            cited.append(did)
    unknown = [c for c in cited if c not in ctx.retrieved]
    if unknown:  
        print(f"[citation-check] removed unretrieved citations {unknown}", file=sys.stderr)
        for c in unknown:
            answer = answer.replace(f" [{c}]", "").replace(f"[{c}]", "")
        cited = [c for c in cited if c not in unknown]
    return {"question_id": qid, "answer": answer, "sources": cited, "forecast_called": ctx.forecast_called}

if __name__ == "__main__":
    import sys
    qs = sys.argv[1:] or ["Q1: Should children in Gulberg go to school on 2026-11-03?"]
    for qq in qs:
        print(json.dumps(ask(qq), indent=2, ensure_ascii=False))


{
  "question_id": "Q-0abbcbcb",
  "answer": "I could not find reliable information on this in the supplied smog documents or forecast, so I won't guess. I can help with smog health advice, school/vehicle/industry rules, and PM2.5 forecasts for the 15 covered Lahore areas.",
  "sources": [],
  "forecast_called": false
}
{
  "question_id": "Q-71335a92",
  "answer": "I could not find reliable information on this in the supplied smog documents or forecast, so I won't guess. I can help with smog health advice, school/vehicle/industry rules, and PM2.5 forecasts for the 15 covered Lahore areas.",
  "sources": [],
  "forecast_called": false
}
{
  "question_id": "Q-0d930bd0",
  "answer": "I could not find reliable information on this in the supplied smog documents or forecast, so I won't guess. I can help with smog health advice, school/vehicle/industry rules, and PM2.5 forecasts for the 15 covered Lahore areas.",
  "sources": [],
  "forecast_called": false
}


## 3. Interactive Test

In [4]:
from pprint import pprint
res = ask('Should children in DHA go to school on 2026-11-06?')
pprint(res)

{'answer': 'Forecast: DHA (sensor S02) on 2026-11-06: predicted daily PM2.5 '
           '168.9 µg/m³ - HAZARDOUS (at or above the 165 µg/m³ threshold). '
           '(Source: team forecast model)\n'
           'School rule for this forecast: half-day timing (close by 12:00 PM) '
           'and suspension of outdoor sports, assemblies and recess [DOC-03].\n'
           '- Decisions are made the evening before, using the next-day '
           "forecast for the school's area. [DOC-03]\n"
           '- Schools close fully when forecast PM2.5 is 250 µg/m³ or above. '
           '[DOC-03]\n'
           '- Schools switch to half-day timing (close by 12:00 PM) when '
           'forecast PM2.5 is 165–249 µg/m³ (hazardous range). [DOC-03]\n'
           '- Children: Keep children indoors during recess and outdoor play '
           'when PM2.5 is forecast at or above 165 µg/m³. [DOC-11]\n'
           '- Children: Watch for increased coughing, wheezing, or fatigue; '
           'consult a pediat